# Emojify：为文本添加表情符号！ 

欢迎来到第二周的第二个作业！你将使用词向量表示来构建表情符号。 
🤩 💫 🔥

你是否曾经想让你的短信更具表现力？你的表情符号应用程序将帮助你做到这一点。 
而不是写：
>“恭喜你升职了！我们喝杯咖啡聊聊吧。爱你！”   

表情符号可以自动将其变成：
>“恭喜升职！👍我们喝杯咖啡聊聊吧。☕️爱你！❤️”

你将实现一个模型，该模型输入一个句子（例如“今晚我们去看棒球比赛！”）并找到与该句子一起使用的最合适的表情符号（⚾️）。

### 使用词向量改进表情符号查找
* 在许多表情符号界面中，你需要记住❤️是“心”符号而不是“爱”符号。 
    * 换句话说，你必须记住输入“heart”才能找到所需的表情符号，而输入“love”不会显示该符号。
* 使用词向量可以制作更灵活的表情界面！
* 使用词向量时，你会发现，即使你的训练集只明确地将几个单词与特定表情符号相关联，你的算法也将能够概括测试集中的其他单词并将其关联到相同的表情符号。
    * 即使这些附加单词甚至没有出现在训练集中，该方法也有效。 
    * 这使你可以构建从句子到表情符号的准确分类器映射，甚至使用小型训练集也是如此。 

### 你将构建什么：
1. 在本练习中，你将从使用词嵌入的基线模型 (Emojifier-V1) 开始。
2. 然后你将构建一个更复杂的模型（Emojifier-V2），进一步包含 LSTM。 

读完本 Notebook，你将能够：

* 在 Keras 中使用预训练词向量创建嵌入层
* 解释GloVe算法的优缺点
* 使用词嵌入构建情感分类器
* 使用 LSTM 构建和训练更复杂的分类器

🏀 👑

👆 😎

（^^^ 表示“技能”的表情符号） 

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/nlp-sequence-models/supplement/qHIve/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [依赖库](#0)
- [1 - 基线模型：Emojifier-V1](#1)
    - [1.1 - 数据集 EMOJISET](#1-1)
    - [1.2 - Emojifier-V1 概述](#1-2)
    - [1.3 - 实现 Emojifier-V1](#1-3)
        - [练习 1 - sentence_to_avg](#ex-1)
    - [1.4 - 实现模型](#1-4)
        - [练习 2 - model](#ex-2)
    - [1.5 - 检查测试集性能](#1-5)
- [2 - Emojifier-V2：在 Keras 中使用 LSTM](#2)
    - [2.1 - 模型概述](#2-1)
    - [2.2 Keras 和小批量](#2-2)
    - [2.3 - 嵌入层](#2-3)
        - [练习 3 - sentences_to_indices](#ex-3)
        - [练习 4 - pretrained_embedding_layer](#ex-4)
    - [2.4 - 构建 Emojifier-V2](#2-4)
        - [练习 5 - Emojify_V2](#ex-5)
    - [2.5 - 训练模型](#2-5)
- [3 - 致谢](#3)

<a name='0'></a>
## 依赖库

让我们开始吧！运行以下单元格来加载你要使用的包。 

In [ ]:
### v3.0

In [ ]:
import numpy as np
from emo_utils import *
import emoji
import matplotlib.pyplot as plt
from test_utils import *

%matplotlib inline

<a name='1'></a>
## 1 - 基线模型：Emojifier-V1

<a name='1-1'></a>
### 1.1 - 数据集 EMOJISET

让我们从构建一个简单的基线分类器开始。 

你有一个很小的数据集（X，Y），其中：
- X 包含 127 个句子（字符串）。
- Y 包含一个 0 到 4 之间的整数标签，对应于每个句子的表情符号。

<img src="images/data_set.png" style="width:700px;height:300px;">
<div style="text-align:center;color:purple;"><b>图 1</b>：EMOJISET - 具有 5 个类别的分类问题。这里给出了一些句子的例子。</div>

使用以下代码加载数据集。数据集分为训练（127 个示例）和测试（56 个示例）。

In [ ]:
X_train, Y_train = read_csv('data/train_emoji.csv')
X_test, Y_test = read_csv('data/tesss.csv')

在下面的代码单元格中，你将找到最大单词数的句子，并将其长度存储在`maxLen`中（*即最长句子中的单词数，稍后将使用*）。让我们分解这段代码以便更好地理解。

- 这里要注意的第一点是 `split()` 将字符串分解为单词列表。因此，如果 `x` 是字符串，则 `len(x.split())` 返回该字符串中的单词数。你可以阅读有关 `split` [这里](https://docs.python.org/3/library/stdtypes.html?highlight=split#str.split) 的更多信息。

- 这里需要注意的第二点是`max` 函数的使用方式。正如可以读到的 [这里](https://docs.python.org/3/library/functions.html#max) 一样，除了一个可迭代对象（在你的例子中是 `X_train`，一个字符串列表）之外，这个函数还具有一个 `key` 参数，该参数可用于修改选择可迭代对象中最大元素的基础。

在本例中，`key` 被选为字符串中的单词数。所以 `max` 函数将返回单词数最多的字符串。

In [ ]:
maxLen = len(max(X_train, key=lambda x: len(x.split())).split())

运行以下单元格打印 X_train 中的句子和 Y_train 中的相应标签。 
* 更改 `idx` 以查看不同的示例。 
* 请注意，由于 iPython Notebook 使用的字体，心形表情符号可能是黑色而不是红色。

In [ ]:
for idx in range(10):
    print(X_train[idx], label_to_emoji(Y_train[idx]))

<a name='1-2'></a>
### 1.2 - Emojifier-V1 概述

在本部分中，你将实现一个名为“Emojifier-v1”的基线模型。  

<center>
<img src="images/image_1.png" style="width:900px;height:300px;">
<div style="text-align:center;color:purple;"><b>图2</b>：基线模型（Emojifier-V1）.</div>
</center>


#### 输入和输出
* 模型的输入是表示一句话的字符串，例如 "I love you"（我爱你）。
* 输出将是形状为 (1,5) 的概率向量（表示有 5 个表情符号可供选择）。
* (1,5) 概率向量被传递到 argmax 层，该层提取概率最高的表情符号的索引。

#### one-hot 编码
* 要将标签转换为适合训练 softmax 分类器的格式，请将 $Y$ 从当前形状 $(m, 1)$ 转换为“one-hot 表示”$(m, 5)$， 
    * 每行都是一个 one-hot 向量，给出一个示例的标签。
    * 这里，`Y_oh` 代表变量名 `Y_oh_train` 和 `Y_oh_test` 中的“Y-one-hot”： 

In [ ]:
Y_oh_train = convert_to_one_hot(Y_train, C = 5)
Y_oh_test = convert_to_one_hot(Y_test, C = 5)

现在，看看 `convert_to_one_hot()` 做了什么。请随意更改 `index` 以打印出不同的值。 

In [ ]:
idx = 50
print(f"Sentence '{X_train[idx]}' has label index {Y_train[idx]}, which is emoji {label_to_emoji(Y_train[idx])}", )
print(f"Label index {Y_train[idx]} in one-hot encoding format is {Y_oh_train[idx]}")

所有数据现在都已准备好输入 Emojify-V1 模型。你已准备好实现该模型！

<a name='1-3'></a>
### 1.3 - 实现 Emojifier-V1

如图2（上图）所示，第一步是：
* 将输入句子中的每个单词转换为其词向量表示形式。
* 取词向量的平均值。 

与本周之前的作业类似，你将使用预训练 50 维 GloVe 嵌入。 

运行以下单元格来加载 `word_to_vec_map`，其中包含所有向量表示。

In [ ]:
word_to_index, index_to_word, word_to_vec_map = read_glove_vecs('data/glove.6B.50d.txt')

你已加载：
- `word_to_index`：从单词到词汇表索引的字典映射 
    - （400,001 个单词，有效索引范围为 0 到 400,000）
- `index_to_word`：从索引到词汇表中对应单词的字典映射
- `word_to_vec_map`：将单词映射到其 GloVe 向量表示的字典。

运行以下单元格检查其是否有效：

In [ ]:
word = "cucumber"
idx = 289846
print("the index of", word, "in the vocabulary is", word_to_index[word])
print("the", str(idx) + "th word in the vocabulary is", index_to_word[idx])

<a name='ex-1'></a>
### 练习 1 - sentence_to_avg

实现`sentence_to_avg()` 

你需要执行两个步骤：

1. 将每个句子转换为小写，然后将句子拆分为单词列表。 
    * `X.lower()` 和 `X.split()` 可能有用。 😉
2. 对于句子中的每个单词，访问其 GloVe 表示形式。
    * 然后取所有这些词向量的平均值。
    * 你可以使用 `numpy.zeros()`，你可以阅读有关 [这里]('https://numpy.org/doc/stable/reference/generated/numpy.zeros.html') 的更多信息。
    
    
#### 附加提示
* 创建时 `avg` 零数组，你会希望它是与另一个形状相同的向量词向量在 `word_to_vec_map`.  
    * 你可以选择 `word_to_vec_map` 中存在的单词并访问其 `.shape` 字段。
    * 请注意不要对你访问的单词进行硬编码。  换句话说，不要假设如果你在 Notebook 内的 `word_to_vec_map` 中看到单词“the”，则当自动评分器调用函数时，该单词将位于 `word_to_vec_map` 中。

**提示**：你可以使用从输入 `sentence` 检索到的任何一个词向量来查找词向量的形状。

In [ ]:
# UNQ_C1 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: sentence_to_avg

def sentence_to_avg(sentence, word_to_vec_map):
    """
    将句子（字符串）转换为单词（字符串）列表。提取每个单词的 GloVe 表示
    并将其值平均化为编码句子含义的单个向量。
    
    参数：
    sentence -- 字符串，来自 X 的一个训练样本
    word_to_vec_map -- 字典将词汇表中的每个单词映射到其 50 维向量表示
    
    返回：
    avg -- 关于句子的平均向量编码信息，形状为 (J,) 的数组，其中 J 可以是任意数字
    """
    # 获取word_to_vec_map中包含的有效单词。 
    any_word = next(iter(word_to_vec_map.keys()))
    
    ### START CODE HERE ###
    # 步骤 1：将句子拆分为小写单词列表（约 1 行）
    words = None

    # 初始化平均值词向量，应该与你的词向量具有相同的形状。
    # 使用 `np.zeros` 并传入任意单词的单词 2 vec 形状的参数
    avg = None
    
    # 初始化计数为0
    count = None
    
    # 步骤 2：对词向量求平均值。你可以循环遍历列表 "words" 中的单词。
    for w in None:
        # 检查word_to_vec_map中是否存在单词
        if w in word_to_vec_map:
            avg += None
            # 增量计数
            count +=1
          
    if count > 0:
        # 求平均值。但前提是 count > 0
        avg = None
    
    ### END CODE HERE ###
    
    return avg

In [ ]:
### 你无法编辑此单元格

# 开始单元测试
avg = sentence_to_avg("Morrocan couscous is my favorite dish", word_to_vec_map)
print("avg = \n", avg)

def sentence_to_avg_test(target):
    # 创建受控词到 vec 映射
    word_to_vec_map = {'a': [3, 3], 'synonym_of_a': [3, 3], 'a_nw': [2, 4], 'a_s': [3, 2], 
                       'c': [-2, 1], 'c_n': [-2, 2],'c_ne': [-1, 2], 'c_e': [-1, 1], 'c_se': [-1, 0], 
                       'c_s': [-2, 0], 'c_sw': [-3, 0], 'c_w': [-3, 1], 'c_nw': [-3, 2]
                      }
    # 将列表转换为 np.arrays
    for key in word_to_vec_map.keys():
        word_to_vec_map[key] = np.array(word_to_vec_map[key])
        
    avg = target("a a_nw c_w a_s", word_to_vec_map)
    assert tuple(avg.shape) == tuple(word_to_vec_map['a'].shape),  "Check the shape of your avg array"  
    assert np.allclose(avg, [1.25, 2.5]),  "Check that you are finding the 4 words"
    avg = target("love a a_nw c_w a_s", word_to_vec_map)
    assert np.allclose(avg, [1.25, 2.5]), "Divide by count, not len(words)"
    avg = target("love", word_to_vec_map)
    assert np.array_equal(avg, [0, 0]), "Average of no words must give an array of zeros"
    avg = target("c_se foo a a_nw c_w a_s deeplearning c_nw", word_to_vec_map)
    assert np.allclose(avg, [0.1666667, 2.0]), "Debug the last example"
    
    print("\033[92mAll tests passed!")
    
sentence_to_avg_test(sentence_to_avg)

# 终端单元测试

<a name='1-4'></a>
### 1.4 - 实现模型

现在你已具备完成 `model()` 函数实现的所有条件！ 
使用`sentence_to_avg()`后你需要：
* 通过前向传播传递平均值
* 计算代价
* 反向传播更新 softmax 参数

<a name='ex-2'></a>
### 练习 2 - model

实现图（2）中描述的`model()` 函数。 

* 你需要在前向传播中实现并计算交叉熵代价的方程如下：
* 变量$Y_{oh}$（“Y one-hot”）是输出标签的one-hot编码。 

$$ z^{(i)} = Wavg^{(i)} + b$$

$$ a^{(i)} = softmax(z^{(i)})$$

$$ \mathcal{L}^{(i)} = - \sum_{k = 0}^{n_y - 1} Y_{oh,k}^{(i)} * log(a^{(i)}_k)$$

**注意**：可以提出更高效的向量化的实现。现在，只需使用嵌套的 for 循环即可更好地理解算法并更轻松地进行调试。

函数 `softmax()` 已提供，并且已导入。

In [ ]:
# UNQ_C2 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: model

def model(X, Y, word_to_vec_map, learning_rate = 0.01, num_iterations = 400):
    """
    用于训练 NumPy 中的词向量表示的模型。
    
    参数：
    X -- 输入数据，NumPy 句子数组作为字符串，形状为 (m,)
    Y -- 标签，NumPy 0到7之间的整数数组，NumPy-形状为(m, 1)的数组
    word_to_vec_map -- 字典将词汇表中的每个单词映射到其 50 维向量表示
    learning_rate -- learning_rate 随机梯度下降算法
    num_iterations -- 迭代次数
    
    返回：
    pred -- 预测向量，NumPy-形状为 (m, 1) 的数组
    W -- softmax 层的权重矩阵，形状为 (n_y, n_h)
    b -- softmax 层的偏置，形状为 (n_y,)
    """
    
    # 获取word_to_vec_map中包含的有效单词 
    any_word = next(iter(word_to_vec_map.keys()))
        
    # 定义训练样本的数量
    m = Y.shape[0]                             # 训练样本的编号
    n_y = len(np.unique(Y))                    # 类别数  
    n_h = word_to_vec_map[any_word].shape[0]   # GloVe 向量的维度 
    
    # 使用Xavier初始化来初始化参数
    W = np.random.randn(n_y, n_h) / np.sqrt(n_h)
    b = np.zeros((n_y,))
    
    # 使用 n_y 类将 Y 转换为 Y_onehot
    Y_oh = convert_to_one_hot(Y, C = n_y) 
    
    # 优化循环
    for t in range(num_iterations): # 循环迭代次数
        
        cost = 0
        dW = 0
        db = 0
        
        for i in range(m):          # 循环训练样本
            
            ### START CODE HERE ### (≈ 4 lines of code)
            # 对第 i 个训练样本中的单词的词向量进行平均
            # 使用你在上面实现的 'sentence_to_avg'  
            avg = None

            # 通过 softmax 层前向传播平均值。 
            # 你可以使用 np.dot() 来执行乘法。
            z = None
            a = None

            # 使用第 i 个训练标签的 one-hot 表示和 "A"（softmax 的输出）添加代价
            cost += None
            ### END CODE HERE ###
            
            # 计算梯度 
            dz = a - Y_oh[i]
            dW += np.dot(dz.reshape(n_y,1), avg.reshape(1, n_h))
            db += dz

            # 使用随机梯度下降更新参数
            W = W - learning_rate * dW
            b = b - learning_rate * db
            
        assert type(cost) == np.float64, "Incorrect implementation of cost"
        assert cost.shape == (), "Incorrect implementation of cost"
        
        if t % 100 == 0:
            print("Epoch: " + str(t) + " --- cost = " + str(cost))
            pred = predict(X, Y, W, b, word_to_vec_map) #预测在emo_utils.py中定义

    return pred, W, b

In [ ]:
### 你无法编辑此单元格

# 单元测试
def model_test(target):
    # 创建受控词到 vec 映射
    word_to_vec_map = {'a': [3, 3], 'synonym_of_a': [3, 3], 'a_nw': [2, 4], 'a_s': [3, 2], 'a_n': [3, 4], 
                       'c': [-2, 1], 'c_n': [-2, 2],'c_ne': [-1, 2], 'c_e': [-1, 1], 'c_se': [-1, 0], 
                       'c_s': [-2, 0], 'c_sw': [-3, 0], 'c_w': [-3, 1], 'c_nw': [-3, 2]
                      }
    # 将列表转换为 np.arrays
    for key in word_to_vec_map.keys():
        word_to_vec_map[key] = np.array(word_to_vec_map[key])
        
    # 训练集。由 a_* 单词组成的句子将属于类别 0，由 c_* 单词组成的句子将属于类别 1
    X = np.asarray(['a a_s synonym_of_a a_n c_sw', 'a a_s a_n c_sw', 'a_s  a a_n', 'synonym_of_a a a_s a_n c_sw', " a_s a_n",
                    " a a_s a_n c ", " a_n  a c c c_e missing",
                   'c c_nw c_n c c_ne', 'c_e c c_se c_s', 'c_nw c a_s c_e c_e', 'c_e a_nw c_sw', 'c_sw c c_ne c_ne'])
    
    Y = np.asarray([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1])
    
    np.random.seed(10)
    pred, W, b = model(X, Y, word_to_vec_map, 0.0025, 110)
    
    assert W.shape == (2, 2), "W must be of shape 2 x 2"
    assert np.allclose(pred.transpose(), Y), "Model must give a perfect accuracy"
    assert np.allclose(b[0], -1 * b[1]), "b should be symmetric in this example"
    
    print("\033[92mAll tests passed!")
    
model_test(model)

运行下一个单元来训练模型并学习 softmax 参数（W，b）。

In [ ]:
np.random.seed(1)
pred, W, b = model(X_train, Y_train, word_to_vec_map)
# print(pred)

太棒了！现在看看它在测试集上的表现如何：

<a name='1-5'></a>
### 1.5 - 检查测试集性能 

请注意，这里使用的`predict` 函数是在`emo_util.py` 中定义的。

In [ ]:
print("Training set:")
pred_train = predict(X_train, Y_train, W, b, word_to_vec_map)
print('Test set:')
pred_test = predict(X_test, Y_test, W, b, word_to_vec_map)

**注意**：
* 假设有 5 个类别，随机猜测的准确率为 20%。 （1/5 = 20%）。
* 经过仅 127 个示例的训练后，这已经是相当不错的性能了。 


#### 模型将表情符号与相关单词相匹配
在训练集中，算法看到了这句话 
>“我爱你。” 

带有标签❤️。 
* 可以检查一下：单词 "treasure"（珍宝）并未出现在训练集中。
* 尽管如此，让我们看看如果你写“我珍惜你”会发生什么。

In [ ]:
X_my_sentences = np.array(["i treasure you", "i love you", "funny lol", "lets play with a ball", "food is ready", "today is not good"])
Y_my_labels = np.array([[0], [0], [2], [1], [4],[3]])

pred = predict(X_my_sentences, Y_my_labels , W, b, word_to_vec_map)
print_predictions(X_my_sentences, pred)

太棒了！ 
* 因为 *treasure* 具有与 *love* 类似的嵌入，所以该算法甚至可以正确地泛化到以前从未见过的单词。 
* *heart*、*dear*、*beloved* 或 *adore* 等词具有与 *love* 类似的嵌入向量。 
    * 请随意修改上面的输入并尝试各种输入句子。 
    * 效果如何？

#### 此模型中未考虑词序
* 请注意，模型无法正确得出以下句子：
>“今天不太好” 

* 该算法忽略词序，因此不擅长理解“不好”之类的短语。 

#### 混淆矩阵
* 打印混淆矩阵还可以帮助了解哪些类别对你的模型来说更困难。 
* 混淆矩阵显示标签为一类（“实际”类）的示例被算法错误标记为不同类（“预测”类）的频率。

打印下面的混淆矩阵：

In [ ]:
# 开始跳过评分
print(Y_test.shape)
print('           '+ label_to_emoji(0)+ '    ' + label_to_emoji(1) + '    ' +  label_to_emoji(2)+ '    ' + label_to_emoji(3)+'   ' + label_to_emoji(4))
print(pd.crosstab(Y_test, pred_test.reshape(56,), rownames=['Actual'], colnames=['Predicted'], margins=True))
plot_confusion_matrix(Y_test, pred_test)
# 评分结束跳过

<div style="color:blue;">

<b>你应该记住：</b>
- 即使只有 127 个训练样本，你也可以获得一个相当不错的表情符号模型。 
    - 这是由于词向量为你提供的泛化能力。 
- Emojify-V1 在诸如 *“这部电影不好而且不令人愉快”* 等句子上表现不佳 
    - 它不理解单词的组合。
    - 它只是将所有单词的嵌入向量平均在一起，而不考虑单词的顺序。 


</div>
    
**不用担心！你将在下一节中构建更好的算法！**

<a name='2'></a>
## 2 - Emojifier-V2：在 Keras 中使用 LSTM 

你将构建一个 LSTM 模型，该模型将单词 **序列** 作为输入！该模型将能够考虑词序。 

Emojifier-V2 将继续使用预训练词嵌入来表示单词。你将词嵌入输入到 LSTM 中，LSTM 将学习预测最合适的表情符号。 

### 依赖库

运行以下单元格来加载你需要的 Keras 包：

In [ ]:
import numpy as np
import tensorflow
np.random.seed(0)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input, Dropout, LSTM, Activation
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.initializers import glorot_uniform
np.random.seed(1)

<a name='2-1'></a>
### 2.1 - 模型概述

这是你将实现的 Emojifier-v2：

<img src="images/emojifier-v2.png" style="width:700px;height:400px;"> <br>
<div style="text-align:center;color:purple;"><b>图3</b>：Emojifier-V2。 2 层 LSTM 序列分类器。</div>

<a name='2-2'></a>
### 2.2 Keras 和小批量 

在本练习中，你想要使用 mini-batch 训练 Keras。然而，大多数深度学习框架要求同一 mini-batch 中的所有序列具有**相同的长度**。 

这就是向量化发挥作用的原因：如果你有一个 3 个单词的句子和一个 4 个单词的句子，那么它们所需的计算是不同的（一个需要 LSTM 的 3 个步骤，一个需要 4 个步骤），所以不可能同时执行这两个句子。
    
#### 填充处理不同长度的序列
* 处理**不同长度**的序列的常见解决方案是使用填充。  具体来说：
    * 设置最大序列长度
    * 填充所有序列以具有相同的长度。 
    
#### 填充示例：
* 给定最大序列长度为 20，你可以用“0”填充每个句子，以便每个输入句子的长度为 20。 
* 因此，句子“我爱你”将表示为 $(e_{I}, e_{love}, e_{you}, \vec{0}, \vec{0}, \ldots, \vec{0})$。 
* 在此示例中，任何超过 20 个单词的句子都必须被截断。 
* 选择最大序列长度的一种方法是仅选择训练集中最长句子的长度。 

<a name='2-3'></a>
### 2.3 - 嵌入层

在Keras中，嵌入矩阵被表示为“层”。

* 嵌入矩阵将单词索引映射到嵌入向量。
    * 单词索引是正整数。
    * 嵌入向量是固定大小的稠密向量。
    * “稠密”向量与稀疏向量相反。这意味着它的大部分值都是非零的。  作为反例，one-hot 编码向量不是“稠密的”。
* 嵌入矩阵可以通过两种方式导出：
    * 训练模型以从头开始导出嵌入。 
    * 使用预训练嵌入。
    
#### 使用和更新预训练嵌入
在本部分中，你将在 Keras 中创建 [Embedding()](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding) 层

* 你将使用 GloVe 50 维向量初始化嵌入层。 
* 在下面的代码中，你将观察 Keras 如何允许你训练或保持该层固定。  
    * 因为你的训练集非常小，所以你将保持 GloVe 嵌入固定而不是更新它们。

#### 嵌入层的输入和输出

* `Embedding()` 层的输入是大小为 **（批量大小，最大输入长度）** 的整数矩阵。 
    * 此输入对应于转换为索引（整数）列表的句子。
    * 输入中的最大整数（最高单词索引）不应大于词汇量大小。
* 嵌入层输出一个形状数组（批量大小、最大输入长度、词向量的维度）。

* 该图显示了两个例句通过嵌入层的传播。 
    * 两个示例均已用零填充至 `max_len=5` 的长度。
    * 词嵌入的长度为 50 个单位。
    * 表示的最终尺寸是 `(2,max_len,50)`。 

<img src="images/embedding1.png" style="width:700px;height:250px;">
<div style="text-align:center;color:purple;"><b>图4</b>：嵌入层</div>

#### 准备输入句子

<a name='ex-3'></a>
### 练习 3 - sentences_to_indices

实现`sentences_to_indices`

这个函数处理句子 X 的数组并将输入返回到嵌入层：

* 将每个训练句子转换为索引列表（索引对应于句子中的每个单词）
* 对所有这些列表进行零填充，使其长度成为最长句子的长度。
    
#### 附加提示：
* 请注意，你可能已考虑在 for 循环中使用 `enumerate()` 函数，但为了传递 AutoGrader，请遵循起始代码，显式初始化并递增 `j`。

In [ ]:
for idx, val in enumerate(["I", "like", "learning"]):
    print(idx, val)

In [ ]:
# UNQ_C3 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: sentences_to_indices

def sentences_to_indices(X, word_to_index, max_len):
    """
    将句子（字符串）数组转换为与句子中的单词对应的索引数组。
    输出形状应能够提供给 `Embedding()`（如图 4 所示）。 
    
    参数：
    X -- 句子（字符串）数组，形状为 (m,)
    word_to_index -- 包含映射到其索引的每个单词的字典
    max_len -- 句子中的最大单词数。你可以假设 X 中的每个句子都不超过这个长度。 
    
    返回：
    X_indices -- 对应于 X 中句子中单词的索引数组，形状为 (m, max_len)
    """
    
    m = X.shape[0]                                   # 训练样本的编号
    
    ### START CODE HERE ###
    # 将 X_indices 初始化为 NumPy 零矩阵和正确的形状（≈ 1 行）
    X_indices = None
    
    for i in range(m):                               # 循环训练样本
        
        # 将第 i 个训练句子转换为小写并将其拆分为单词。你应该得到一个单词列表。
        sentence_words = None
        
        # 初始化j为0
        j = None
        
        # 循环sentence_words的单词

        for w in None:
            # 如果 w 存在于 word_to_index 字典中
            if w in None:
                # 将 X_indices 的第 (i,j) 条目设置为正确单词的索引。
                X_indices[i, j] = None
                # 将 j 增加到 j + 1
                j =  None
            
    ### END CODE HERE ###
    
    return X_indices

In [ ]:
### 你无法编辑此单元格

# 单元测试
def sentences_to_indices_test(target):
    
    # 创建 word_to_index 字典
    word_to_index = {}
    for idx, val in enumerate(["i", "like", "learning", "deep", "machine", "love", "smile", '´0.=']):
        word_to_index[val] = idx + 1;
       
    max_len = 4
    sentences = np.array(["I like deep learning", "deep ´0.= love machine", "machine learning smile", "$"]);
    indexes = target(sentences, word_to_index, max_len)
    print(indexes)
    
    assert type(indexes) == np.ndarray, "Wrong type. Use np arrays in the function"
    assert indexes.shape == (sentences.shape[0], max_len), "Wrong shape of ouput matrix"
    assert np.allclose(indexes, [[1, 2, 4, 3],
                                 [4, 8, 6, 5],
                                 [5, 3, 7, 0],
                                 [0, 0, 0, 0]]), "Wrong values. Debug with the given examples"
    
    print("\033[92mAll tests passed!")
    
sentences_to_indices_test(sentences_to_indices)

**预期值**

```
[[1. 2. 4. 3.]
 [4. 8. 6. 5.]
 [5. 3. 7. 0.]
 [0. 0. 0. 0.]]
```

运行以下单元格来检查 `sentences_to_indices()` 的作用，并查看结果。

In [ ]:
X1 = np.array(["funny lol", "lets play baseball", "food is ready for you"])
X1_indices = sentences_to_indices(X1, word_to_index, max_len=5)
print("X1 =", X1)
print("X1_indices =\n", X1_indices)

#### 构建嵌入层

现在，你将使用预训练词向量在 Keras 中构建 `Embedding()` 层。 

* 嵌入层将单词索引列表作为输入。
    * `sentences_to_indices()` 创建这些单词索引。
* 嵌入层将返回句子的词嵌入。 

<a name='ex-4'></a>
### 练习 4 - pretrained_embedding_layer

通过以下步骤实现 `pretrained_embedding_layer()`：

1. 将嵌入矩阵初始化为 NumPy 零数组。
    * 嵌入矩阵对于词汇表中的每个唯一单词都有一行。
        * 还有一行用于处理“未知”单词。
        * 所以 vocab_size 是唯一单词的数量加一。
    * 每行将存储一个单词的向量表示。 
        * 例如，如果使用 GloVe 词向量，一行可能有 50 个位置长。
    * 在下面的代码中，`emb_dim` 代表词嵌入的长度。
2. 用单词的向量表示填充嵌入矩阵的每一行
    * `word_to_index` 中的每个单词都是一个字符串。
    * word_to_vec_map 是一个字典，其中键是字符串，值是词向量。
3. 定义 Keras 嵌入层。 
    * 使用[Embedding()](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding)。 
    * 输入维度等于词汇长度（唯一单词的数量加一）。
    * 输出维度等于词嵌入中的位置数。
    * 固定该层的嵌入。
        * 如果你要设置`trainable = True`，那么它将允许优化算法修改词嵌入的值。
        * 在这种情况下，你不希望模型修改词嵌入。
4. 将嵌入权重设置为等于嵌入矩阵。
    * 请注意，这部分代码已经为你完成，不需要修改！ 

In [ ]:
# UNQ_C4 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: pretrained_embedding_layer

def pretrained_embedding_layer(word_to_vec_map, word_to_index):
    """
    创建 Keras Embedding() 层并加载到预训练 GloVe 50 维向量中。
    
    参数：
    word_to_vec_map -- 将单词映射到其 GloVe 向量表示的字典。
    word_to_index -- 从单词到词汇表索引的字典映射（400,001 个单词）

    返回：
    embedding_layer -- 预训练层 Keras 实例
    """
    
    vocab_size = len(word_to_index) + 1              # 添加 1 以适应 Keras 嵌入（必需）
    any_word = next(iter(word_to_vec_map.keys()))
    emb_dim = word_to_vec_map[any_word].shape[0]    # 定义 GloVe 词向量的维度 (= 50)
      
    ### START CODE HERE ###
    # 步骤 1
    # 将嵌入矩阵初始化为 NumPy 零数组。
    # 请参阅上面的说明以选择正确的形状。
    emb_matrix = None
    
    # 步骤 2
    # 将嵌入矩阵的每一行 "idx" 设置为 
    # 词汇表中第 idx 个单词的词向量表示
    for word, idx in word_to_index.items():
        emb_matrix[idx, :] = None

    # 步骤 3
    # 使用正确的输入和输出大小定义 Keras 嵌入层
    # 使其不可训练。
    embedding_layer = None
    ### END CODE HERE ###

    # 步骤4（已为你完成；请勿修改）
    # 构建嵌入层，在设置嵌入层的权重之前需要这样做。 
    embedding_layer.build((None,)) # 请勿修改"None"。  这行代码按原样完整。
    
    # 将嵌入层的权重设置为嵌入矩阵。你的层现在是预训练。
    embedding_layer.set_weights([emb_matrix])
    
    return embedding_layer

In [ ]:
### 你无法编辑此单元格

# 单元测试
def pretrained_embedding_layer_test(target):
    # 创建受控词到 vec 映射
    word_to_vec_map = {'a': [3, 3], 'synonym_of_a': [3, 3], 'a_nw': [2, 4], 'a_s': [3, 2], 'a_n': [3, 4], 
                       'c': [-2, 1], 'c_n': [-2, 2],'c_ne': [-1, 2], 'c_e': [-1, 1], 'c_se': [-1, 0], 
                       'c_s': [-2, 0], 'c_sw': [-3, 0], 'c_w': [-3, 1], 'c_nw': [-3, 2]
                      }
    # 将列表转换为 np.arrays
    for key in word_to_vec_map.keys():
        word_to_vec_map[key] = np.array(word_to_vec_map[key])
        
    # 创建 word_to_index 字典
    word_to_index = {}
    for idx, val in enumerate(list(word_to_vec_map.keys())):
        word_to_index[val] = idx;
        
    np.random.seed(1)
    embedding_layer = target(word_to_vec_map, word_to_index)
    
    assert type(embedding_layer) == Embedding, "Wrong type"
    assert embedding_layer.input_dim == len(list(word_to_vec_map.keys())) + 1, "Wrong input shape"
    assert embedding_layer.output_dim == len(word_to_vec_map['a']), "Wrong output shape"
    assert np.allclose(embedding_layer.get_weights(), 
                       [[[ 3, 3], [ 3, 3], [ 2, 4], [ 3, 2], [ 3, 4],
                       [-2, 1], [-2, 2], [-1, 2], [-1, 1], [-1, 0],
                       [-2, 0], [-3, 0], [-3, 1], [-3, 2], [ 0, 0]]]), "Wrong vaulues"
    print("\033[92mAll tests passed!")
       
    
pretrained_embedding_layer_test(pretrained_embedding_layer)

In [ ]:
embedding_layer = pretrained_embedding_layer(word_to_vec_map, word_to_index)
print("weights[0][1][1] =", embedding_layer.get_weights()[0][1][1])
print("Input_dim", embedding_layer.input_dim)
print("Output_dim",embedding_layer.output_dim)

<a name='2-4'></a>
### 2.4 - 构建 Emojifier-V2

现在你已准备好构建 Emojifier-V2 模型，在其中将嵌入层的输出输入到 LSTM 网络！

<img src="images/emojifier-v2.png" style="width:700px;height:400px;"> <br>
<div style="text-align:center;color:purple;"><b>图3</b>：Emojifier-v2。 2 层 LSTM 序列分类器。</div> 


<a name='ex-5'></a>
### 练习 5 - Emojify_V2

实现`Emojify_V2()`

这个函数构建了图(3)所示架构的Keras图。 

* 该模型将由 `input_shape` 定义的形状为 (`m`, `max_len`, ) 的句子数组作为输入。 
* 该模型输出形状为 (`m`, `C = 5`) 的 softmax 概率向量。 

* 你可能需要使用以下 Keras 层：
    * [Input()](https://www.tensorflow.org/api_docs/python/tf/keras/Input)
        * 设置`shape`和`dtype`参数。
        * 输入是整数，因此你可以将数据类型指定为字符串“int32”。
    * [LSTM()](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM)
        * 设置`units`和`return_sequences`参数。
    * [Dropout()](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout)
        * 设置`rate`参数。
    * [Dense()](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)
        * 设置`units`， 
        * 请注意，`Dense()` 有一个 `activation` 参数。  为了通过AutoGrader，请勿将激活值设置在`Dense()`内。  使用单独的 `Activation` 层来执行此操作。
    * [Activation()](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Activation)
        * 你可以将你选择的激活值作为小写字符串传递。
    * [Model()](https://www.tensorflow.org/api_docs/python/tf/keras/Model)
        * 设置`inputs`和`outputs`。


#### 附加提示
* 请记住，这些 Keras 层返回一个对象，你将把前一层的输出作为该对象的输入参数。  返回的对象可以在同一行中创建和调用。

```Python
# 如何用两行代码使用Keras层
dense_object = Dense(units = ...)
X = dense_object(inputs)

# 如何在一行代码中使用Keras层
X = Dense(units = ...)(inputs)
```

* `pretrained_embedding_layer` 返回的 `embedding_layer` 是一个层对象，可以调用函数，传入单个参数（句子索引）。

* 这是一些示例代码，以防你遇到困难：😊
```Python
raw_inputs = Input(shape=(maxLen,), dtype='int32')
preprocessed_inputs = ... # 一些预处理
X = LSTM(units = ..., return_sequences= ...)(processed_inputs)
X = Dropout(rate = ..., )(X)
...
X = Dense(units = ...)(X)
X = Activation(...)(X)
model = Model(inputs=..., outputs=...)
...
```

In [ ]:
# UNQ_C5 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: Emojify_V2

def Emojify_V2(input_shape, word_to_vec_map, word_to_index):
    """
    函数创建 Emojify-v2 模型的图表。
    
    参数：
    input_shape -- 输入的形状，通常为 (max_len,)
    word_to_vec_map -- 字典将词汇表中的每个单词映射到其 50 维向量表示
    word_to_index -- 从单词到词汇表索引的字典映射（400,001 个单词）

    返回：
    model -- Keras 中的模型实例
    """
    
    ### START CODE HERE ###
    # 定义 sentence_indices 作为图的输入。
    # 它的形状应该是 input_shape ，dtype 是 'int32' （因为它包含索引，它们是整数）。
    sentence_indices = None
    
    # 使用 GloVe 向量创建嵌入层预训练（约 1 行）
    embedding_layer = None
    
    # 通过嵌入层传播 sentence_indices
    # （请参阅说明中的其他提示）。
    embeddings = None   
    
    # 通过具有 128 维隐藏状态的 LSTM 层传播嵌入
    # 返回的输出应该是一批序列。
    X = None
    # 以 0.5 的概率添加 Dropout
    X = None 
    # 通过另一个具有 128 维隐藏状态的 LSTM 层传播 X
    # 返回的输出应该是单个隐藏状态，而不是一批序列。
    X = None
    # 以 0.5 的概率添加 Dropout
    X = None 
    # 通过 5 个单元的全连接层传播 X
    X = None
    # 添加 softmax 激活函数
    X = None
    
    # 创建将 sentence_indices 转换为 X 的 Model 实例。
    model = None
    
    ### END CODE HERE ###
    
    return model

In [ ]:
### 你无法编辑此单元格

# 单元测试
from tensorflow.python.keras.engine.functional import Functional

def Emojify_V2_test(target):
    # 创建受控词到 vec 映射
    word_to_vec_map = {'a': [3, 3], 'synonym_of_a': [3, 3], 'a_nw': [2, 4], 'a_s': [3, 2], 'a_n': [3, 4], 
                       'c': [-2, 1], 'c_n': [-2, 2],'c_ne': [-1, 2], 'c_e': [-1, 1], 'c_se': [-1, 0], 
                       'c_s': [-2, 0], 'c_sw': [-3, 0], 'c_w': [-3, 1], 'c_nw': [-3, 2]
                      }
    # 将列表转换为 np.arrays
    for key in word_to_vec_map.keys():
        word_to_vec_map[key] = np.array(word_to_vec_map[key])
        
    # 创建 word_to_index 字典
    word_to_index = {}
    for idx, val in enumerate(list(word_to_vec_map.keys())):
        word_to_index[val] = idx;
        
    maxLen = 4
    model = target((maxLen,), word_to_vec_map, word_to_index)
    
    assert type(model) == Functional, "Make sure you have correctly created Model instance which converts \"sentence_indices\" into \"X\""
    
    expectedModel = [['InputLayer', [(None, 4)], 0], ['Embedding', (None, 4, 2), 30], ['LSTM', (None, 4, 128), 67072, (None, 4, 2), 'tanh', True], ['Dropout', (None, 4, 128), 0, 0.5], ['LSTM', (None, 128), 131584, (None, 4, 128), 'tanh', False], ['Dropout', (None, 128), 0, 0.5], ['Dense', (None, 5), 645, 'linear'], ['Activation', (None, 5), 0]]
    comparator(summary(model), expectedModel)
    
    
Emojify_V2_test(Emojify_V2)

运行以下单元格来创建模型并检查其摘要。 

* 由于数据集中的所有句子都少于 10 个单词，因此选择 `max_len = 10`。  
* 你应该看到你的架构使用 20,223,927 个参数，其中 20,000,050 个（词嵌入）是不可训练的，其余 223,877 个是可训练的。 
* 因为你的词汇量有 400,001 个单词（有效索引从 0 到 400,000），所以有 400,001\*50 = 20,000,050 个不可训练参数。 

In [ ]:
model = Emojify_V2((maxLen,), word_to_vec_map, word_to_index)
model.summary()

#### 编译模型 

像往常一样，在 Keras 中创建模型后，你需要编译它并定义要使用的损失、优化器和指标。使用 `categorical_crossentropy` 损失、`adam` 优化器和 `['accuracy']` 指标编译模型：

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

<a name='2-5'></a>
### 2.5 - 训练模型 

现在开始训练模型！Emojifier-V2 的 `model` 接收形状为 (`m`, `max_len`) 的数组，输出形状为 (`m`, `number of classes`) 的类别概率数组。因此，需要将句子字符串数组 X_train 转换为词索引数组 X_train_indices，并将类别索引标签 Y_train 转换为 one-hot 标签 Y_train_oh。

In [ ]:
X_train_indices = sentences_to_indices(X_train, word_to_index, maxLen)
Y_train_oh = convert_to_one_hot(Y_train, C = 5)

使用 `epochs = 50` 和 `batch_size = 32` 将 Keras 模型安装到 `X_train_indices` 和 `Y_train_oh` 上。

In [ ]:
model.fit(X_train_indices, Y_train_oh, epochs = 50, batch_size = 32, shuffle=True)

你的模型在训练集上的准确率应在 **90% 到 100% 之间**。确切的模型精度可能会有所不同！ 

运行以下单元格以在测试集上评估你的模型： 

In [ ]:
X_test_indices = sentences_to_indices(X_test, word_to_index, max_len = maxLen)
Y_test_oh = convert_to_one_hot(Y_test, C = 5)
loss, acc = model.evaluate(X_test_indices, Y_test_oh)
print()
print("Test accuracy = ", acc)

你应该获得 80% 到 95% 之间的测试准确率。运行下面的单元格以查看标签错误的示例： 

In [ ]:
# 此代码允许你查看错误标记的示例
C = 5
y_test_oh = np.eye(C)[Y_test.reshape(-1)]
X_test_indices = sentences_to_indices(X_test, word_to_index, maxLen)
pred = model.predict(X_test_indices)
for i in range(len(X_test)):
    x = X_test_indices
    num = np.argmax(pred[i])
    if(num != Y_test[i]):
        print('Expected emoji:'+ label_to_emoji(Y_test[i]) + ' prediction: '+ X_test[i] + label_to_emoji(num).strip())

现在你可以在自己的示例中尝试一下！下面写下你自己的句子：

In [ ]:
# 更改下面的句子以查看你的预测。确保所有单词都在 Glove 嵌入中。  
x_test = np.array(['I cannot play'])
X_test_indices = sentences_to_indices(x_test, word_to_index, maxLen)
print(x_test[0] +' '+  label_to_emoji(np.argmax(model.predict(X_test_indices))))

#### LSTM 版本能够考虑词序
* Emojify-V1 模型没有正确预测“不开心”，但你的 Emojify-V2 实现是正确的！ 
    * 如果没有，请注意 Keras 的输出每次都稍微随机，所以这可能就是原因。 
* 当前模型在理解否定（例如“不高兴”）方面仍然不是很稳健
    * 这是因为训练集很小，没有很多否定的例子。 
    * 如果训练集更大，LSTM 模型在理解更复杂的句子方面会比 Emojify-V1 模型好得多。 

### 恭喜！
 
你已经完成了这个 Notebook，并利用 LSTM 的力量让你的话语更加富有情感！ ❤️❤️❤️

到目前为止，你已经： 

* 创建了一个嵌入矩阵
* 观察负采样如何比其他方法更有效地学习词向量
* 体验了GloVe算法的优缺点
* 并使用词嵌入构建了一个情感分类器！ 

酷！ （或表情符号：😎😎😎） 

<div style="color:blue;">

<b>你应该记住什么</b>：
- 如果你有一个训练集很小的 NLP 任务，那么使用词嵌入可以显著帮助你的算法。 
- 词嵌入允许你的模型处理测试集中甚至可能不会出现在训练集中的单词。 
- Keras（以及大多数其他深度学习框架）中的训练序列模型需要一些重要的细节：
    - 要使用 mini-batch，需要对序列进行 **填充**，以便 mini-batch 中的所有示例具有 **相同的长度**。 
    - `Embedding()` 层可以使用预训练值进行初始化。 
        - 这些值可以是固定的，也可以在你的数据集上进一步训练。 
        - 然而，如果你的标记数据集很小，通常不值得尝试训练大量预训练嵌入集。   
    - `LSTM()` 有一个名为 `return_sequences` 的标志，用于决定是否要返回所有隐藏状态还是仅返回最后一个。 
    - 你可以在 `LSTM()` 之后立即使用 `Dropout()` 来规范你的网络。

</div>


### 输入句子：
```Python
"Congratulations on finishing this assignment and building an Emojifier."
"We hope you're happy with what you've accomplished in this notebook!"
```
### 输出表情符号：
# 😀😀😀😀😀😀

☁ 👋🚀 ☁☁

      ✨ 再见！
      
☁ ✨  🎈

      ✨  ☁
  
         ✨
 
     ✨
 
🌾✨💨 🏃 🏠🏢                    

<a name='3'></a>
## 3 - 致谢

感谢 Alison Darcy 和 Woebot 团队为创建此作业提供的建议。 
* Woebot 是一个聊天机器人朋友，随时准备与你交谈 24/7。 
* Woebot 的部分技术使用词嵌入来理解你所说的话的情绪。 
* 你可以通过http://woebot.io与Woebot聊天

<img src="images/woebot.png" style="width:600px;height:300px;">